In [1]:
import pandas as pd
from tqdm import tqdm
import os
import yaml

import sys
sys.path.append('..')
from mlrose_hiive import TSPGenerator
from mlrose_hiive import SARunner
import mlrose_hiive

In [2]:
SEED = 0
TEMPERATURE_LIST = [1e0, 1e1, 1e2, 1e3, 1e4]
DECAY_LIST = {
    'GeomDecay': mlrose_hiive.GeomDecay, 
    # 'ExpDecay': mlrose_hiive.ExpDecay, 
    # 'ArithDecay': mlrose_hiive.ArithDecay,
}

PROBLEM_SIZE_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['problem_size']]
ITERATIONS_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['iterations']]
MAX_ATTEMPTS_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['attempts']]
NUM_RUNS = yaml.load(open('parameters.yml'), yaml.Loader)['num_runs']
POPULATION_SIZE_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['population_size']]

assert(len(PROBLEM_SIZE_LIST) == 1)
EXPERIMENT_NAME = f'TSP_SA'

In [3]:
output_dir = f'metrics_size={PROBLEM_SIZE_LIST[0]}_iters={ITERATIONS_LIST[0]}_pop={POPULATION_SIZE_LIST[0]}_attempts={MAX_ATTEMPTS_LIST[0]}/{EXPERIMENT_NAME}'
os.makedirs(output_dir, exist_ok=True)

In [4]:
all_df = pd.DataFrame()
group_i = 0
run_i = 0
for problem_size in PROBLEM_SIZE_LIST:
    print(f'Problem Size: {problem_size}')
    for max_iterations in ITERATIONS_LIST:
        print(f'Max Iterations: {max_iterations}')
        for max_attempts in MAX_ATTEMPTS_LIST:
            print(f'Max Attempts: {max_attempts}')
            for start_temperature in TEMPERATURE_LIST:
                print(f"Start Temperature: {start_temperature}")
                for decay_str, decay_cls in DECAY_LIST.items():
                    print(f"Decay type: {decay_str}")
                    for i in tqdm(range(NUM_RUNS)):
                        problem = TSPGenerator().generate(seed=SEED+i, number_of_cities=problem_size)
                        sa = SARunner(
                            problem=problem,
                            experiment_name='dontcare',
                            output_directory='./',
                            seed=SEED+i,
                            iteration_list=[max_iterations],
                            max_attempts=max_attempts,
                            temperature_list=[start_temperature],
                            decay_list=[decay_cls],
                        )
                        _, df_run_curves = sa.run()
                        df_run_curves['group_number'] = group_i
                        df_run_curves['run_number'] = run_i
                        df_run_curves['problem_size'] = problem_size
                        df_run_curves['max_iterations'] = max_iterations
                        df_run_curves['max_attempts'] = max_attempts
                        df_run_curves['start_temperature'] = start_temperature
                        df_run_curves['decay_type'] = decay_str

                        print(f"Max Fitness: {df_run_curves['Fitness'].max()}")
                        print(f"Max Iteration: {df_run_curves['Iteration'].max()}")

                        all_df = pd.concat([all_df, df_run_curves], axis=0)
                        run_i += 1
                    group_i += 1
all_df.reset_index(inplace=True, drop=True)

Problem Size: 10
Max Iterations: 10
Max Attempts: 1000
Start Temperature: 1.0
Decay type: GeomDecay


100%|██████████| 3/3 [00:00<00:00, 118.82it/s]


Max Fitness: 1298.551899183571
Max Iteration: 10
Max Fitness: 1068.1483700349288
Max Iteration: 10
Max Fitness: 1400.9252885718133
Max Iteration: 10
Start Temperature: 10.0
Decay type: GeomDecay


100%|██████████| 3/3 [00:00<00:00, 123.87it/s]


Max Fitness: 1298.551899183571
Max Iteration: 10
Max Fitness: 1068.1483700349288
Max Iteration: 10
Max Fitness: 1400.9252885718133
Max Iteration: 10
Start Temperature: 100.0
Decay type: GeomDecay


...

Max Fitness: 1298.551899183571
Max Iteration: 10
Max Fitness: 1157.3181282767596
Max Iteration: 10
Max Fitness: 1428.30799023879
Max Iteration: 10
Start Temperature: 1000.0
Decay type: GeomDecay


100%|██████████| 3/3 [00:00<00:00, 146.51it/s]


Max Fitness: 1510.6798605026215
Max Iteration: 10
Max Fitness: 1414.6988060037797
Max Iteration: 10
Max Fitness: 1649.6139342757792
Max Iteration: 10
Start Temperature: 10000.0
Decay type: GeomDecay


100%|██████████| 3/3 [00:00<00:00, 148.55it/s]

Max Fitness: 1510.6798605026215
Max Iteration: 10
Max Fitness: 1411.5030404919626
Max Iteration: 10
Max Fitness: 1649.6139342757792
Max Iteration: 10


In [5]:
print(f"Max: {all_df['Fitness'].max()}")
print(f"Min: {all_df['Fitness'].min()}")

Max: 1649.6139342757792
Min: 955.5712369845377


In [6]:
all_df.to_csv(os.path.join(output_dir, 'learning_curve.csv'), index=False)